## SETUP LOGFIRE

In [1]:
import os, time, warnings
warnings.filterwarnings("ignore")


import logfire 
from dotenv import load_dotenv

load_dotenv()


# Verify keys
print("LOGFIRE_TOKEN  :", "✅" if os.getenv("LOGFIRE_TOKEN")  else "❌  missing")
print("GROQ_API_KEY   :", "✅" if os.getenv("GROQ_API_KEY")   else "❌  missing")
print("GEMINI_API_KEY :", "✅" if os.getenv("GEMINI_API_KEY") else "❌  missing")

LOGFIRE_TOKEN  : ✅
GROQ_API_KEY   : ✅
GEMINI_API_KEY : ✅


In [2]:
import logfire

logfire.configure()
logfire.info('hello , {place}!', place = 'world')

Logfire project URL: https://logfire-us.pydantic.dev/sahilsawant7120/logfire-demo

00:59:34.773 hello , world!


## Simple INFO

In [3]:
logfire.info("notebook_started",
              notebook_name = "pydantic_logfire.ipynb",
              timestamp = time.time(),
              part = "part_1",
              student = "Sahil Sawant",
              tool = "Pydantic Logfire"
              )

01:03:52.874 notebook_started


## Implementing TRACE

In [4]:
with logfire.span("data_processing_simulation", dataset="llm_course", rows=1000):
    logfire.info("step_started", step=1, action="loading data")
    time.sleep(0.3)

    logfire.info("step_started", step=2, action="transforming", columns=12)
    time.sleep(0.2)

    logfire.info("step_started", step=3, action="saving results", output="/tmp/out.csv")

01:07:34.466 data_processing_simulation
01:07:34.466   step_started
01:07:34.767   step_started
01:07:34.968   step_started


## Experiment 2 — Structured Logging with Pydantic Models

In [5]:
from pydantic import BaseModel
from typing import Optional

class LLMRequest(BaseModel):
    user_id: str
    session_id: str
    query: str
    model : str
    temperature: float = 0.7
    max_tokens: Optional[int] = None

class LLMResponse(BaseModel):
    answer: str
    input_tokens: int
    output_tokens: int
    latency_ms : float
    model_used: str

In [7]:
#Passing DATA

request = LLMRequest(
    user_id="user_123",
    session_id="session_456",
    query="What is the capital of France?",
    model="gpt-4",
    temperature=0.5,
    max_tokens=100
)

with logfire.span("llm_request",
                user_id=request.user_id,
                session_id=request.session_id,
                model_used=request.model 
                ):

    logfire.info("request_received" , **request.model_dump())

    time.sleep(0.1)

    response = LLMResponse(
        answer="The capital of France is Paris.",
        input_tokens=10,
        output_tokens=7,
        latency_ms=120.5,
        model_used="llm-4"
    )
    logfire.info("response_generated" , **response.model_dump())

print(response)


01:25:41.679 llm_request
01:25:41.680   request_received
01:25:41.781   response_generated
answer='The capital of France is Paris.' input_tokens=10 output_tokens=7 latency_ms=120.5 model_used='llm-4'
